In [1]:
"""
Models to try:

- LightGBM
"""

'\nModels to try:\n\n- LightGBM\n'

In [2]:
import pandas as pd


In [3]:
"""
Features:

 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          2111 non-null   object 
 1   Age                             2111 non-null   float64
 2   family_history_with_overweight  2111 non-null   object 
 3   FAVC                            2111 non-null   object 
 4   FCVC                            2111 non-null   float64
 5   NCP                             2111 non-null   float64
 6   CAEC                            2111 non-null   object 
 7   SMOKE                           2111 non-null   object 
 8   CH2O                            2111 non-null   float64
 9   SCC                             2111 non-null   object 
 10  FAF                             2111 non-null   float64
 11  TUE                             2111 non-null   float64
 12  CALC                            2111 non-null   object 
 13  MTRANS                          2111 non-null   object 
 14  NObeyesdad                      2111 non-null   object 
"""

# Keep only specified features and remove Unnamed and Height/Weight columns
COLUMNS_TO_KEEP = [
    'Gender', 'Age', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP',
    'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'
]

NUMERICAL_FEATURES = ['Age'] #, 'Height', 'Weight']
BOOLEAN_FEATURES = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
CATEGORICAL_FEATURES = ['CAEC', 'CALC', 'MTRANS']
INTEGER_FEATURES = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']



In [4]:
# Load the datasets

raw_real_data = pd.read_csv("/work/datasets/real-data-20250501-154339.csv")
raw_synthetic_data = pd.read_csv("/work/datasets/synth_clean_20.csv")

RAW_DATA = {
    'real': raw_real_data,
    'synthetic': raw_synthetic_data
}

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# yes/no to 1/0
class BooleanToBinaryTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.replace({'yes': 1, 'no': 0})

# made a custom mapper to have control over value
class GenderToBinaryTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='Gender'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = X[self.column].map({'Male': 0, 'Female': 1})
        return X


# Round all numerical features, for the messed up values 
round_only = FunctionTransformer(np.round, validate=False)

# Define transformers Pipelines
numerical_transformer = Pipeline([
    ('scaler', StandardScaler())
])

boolean_transformer = Pipeline([
    ('boolean_to_binary', BooleanToBinaryTransformer())
])

categorical_transformer = Pipeline([
    ('onehotencoder',OneHotEncoder(drop='first', sparse=False))
])

# Combine all transformers into one preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, NUMERICAL_FEATURES),
        ('bool', boolean_transformer, BOOLEAN_FEATURES),
        ('gender', GenderToBinaryTransformer(), ['Gender']),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough' # Leave the rest of the columns as they are
)

In [6]:
#TODO update
def get_final_feature_names(preprocessor, X_df, numerical_features, boolean_features, categorical_features):
    """
    Reconstructs the full list of feature names after transformation.

    Parameters:
        preprocessor: the fitted ColumnTransformer
        X_df: the original unprocessed DataFrame (e.g., X_train)
        numerical_features, boolean_features, categorical_features: lists of assigned features

    Returns:
        A list of final feature names in the order they appear in the transformed array
    """

    #Assigned features
    assigned_features = numerical_features + boolean_features + categorical_features

    #Passthrough features (not transformed)
    all_features = list(X_df.columns)
    passthrough_features = [f for f in all_features if f not in assigned_features]

    #Get the feature names from OneHotEncoder
    cat_ohe = preprocessor.named_transformers_['cat']['onehotencoder']
    cat_feature_names = cat_ohe.get_feature_names_out(categorical_features)

    #Combine all
    final_feature_names = numerical_features + boolean_features + list(cat_feature_names) + passthrough_features

    return final_feature_names

In [7]:
from sklearn.model_selection import train_test_split

def preprocess(df):
    # Drop unnecessary columns
    df = df[COLUMNS_TO_KEEP]

    # Preprocess the data
    X = df.drop(columns=['NObeyesdad'])
    y = df['NObeyesdad']

    X = pd.DataFrame(preprocessor.fit_transform(X), columns=get_final_feature_names(preprocessor, X, NUMERICAL_FEATURES, BOOLEAN_FEATURES, CATEGORICAL_FEATURES))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

In [11]:
# Check if features are equal, otherwise insert 0 in place for missing features
PROCESSED_DATA = {
    'real': preprocess(raw_real_data),
    'synthetic': preprocess(raw_synthetic_data)
}

# Get all unique columns from both datasets
all_columns = set(PROCESSED_DATA['real']['X_train'].columns).union(set(PROCESSED_DATA['synthetic']['X_train'].columns))

# Add missing columns with zeros to both datasets
for dataset_type in ['real', 'synthetic']:
    for split in ['X_train', 'X_test']:
        missing_cols = all_columns - set(PROCESSED_DATA[dataset_type][split].columns)
        for col in missing_cols:
            PROCESSED_DATA[dataset_type][split][col] = 0
            
        # Ensure columns are in the same order
        PROCESSED_DATA[dataset_type][split] = PROCESSED_DATA[dataset_type][split][sorted(all_columns)]

# Verify that columns are now equal
assert PROCESSED_DATA['real']['X_train'].columns.equals(PROCESSED_DATA['synthetic']['X_train'].columns)
print("Features are now aligned between real and synthetic datasets")

Features are now aligned between real and synthetic datasets


In [14]:
# Write the processed data to CSV

PATH_PREFIX = '/work/datasets/preprocessed/'

for name, data in PROCESSED_DATA.items():
    data['X_train'].to_csv(PATH_PREFIX + name + '_X_train.csv', index=False)
    data['X_test'].to_csv(PATH_PREFIX + name + '_X_test.csv', index=False)
    data['y_train'].to_csv(PATH_PREFIX + name + '_y_train.csv', index=False)
    data['y_test'].to_csv(PATH_PREFIX + name + '_y_test.csv', index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a441f35e-4b4c-4c50-b56a-1aea6b800ed8' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>